#### CORRELATION PROFILE

In [0]:
%run "./02_DATA_PREPARATION"

In [0]:
# ============================================================
# 09.3 PARÂMETROS DE EXECUÇÃO
# ============================================================
#
# O QUE FAZ:
# Define limites e escolhas dos métodos.
#
# COMO FAZ:
# Centraliza parâmetros de custo e interpretação.
#
# POR QUE É IMPORTANTE:
# Permite adaptar a análise a diferentes datasets.
#
# PERGUNTA RESPONDIDA:
# "Como executar com controle de custo?"

CORRELATION_MAX_COLUMNS = 12
CORRELATION_MIN_VALID_PAIRS = 3
CORRELATION_ENABLE_SPEARMAN = False  # Habilite apenas após avaliar custo de ordenação.
CORRELATION_STRONG_THRESHOLD = 0.80
CORRELATION_WEAK_THRESHOLD = 0.30
CORRELATION_EXCLUDED_COLUMNS = []  # Ex.: identificadores numéricos sem significado estatístico.
CORRELATION_DISPLAY_RESULTS = bool(globals().get("DISPLAY_RESULTS", True))

assert CORRELATION_MAX_COLUMNS >= 2
assert CORRELATION_MIN_VALID_PAIRS >= 2
assert 0 <= CORRELATION_WEAK_THRESHOLD < CORRELATION_STRONG_THRESHOLD <= 1

In [0]:
# ============================================================
# 09.4 SELEÇÃO DE COLUNAS ANALISÁVEIS
# ============================================================
#
# O QUE FAZ:
# Seleciona apenas colunas numéricas e não booleanas.
#
# COMO FAZ:
# Intersecciona colunas_correlation com o schema real.
#
# POR QUE É IMPORTANTE:
# Evita correlação indevida de campos não numéricos.
#
# PERGUNTA RESPONDIDA:
# "Quais colunas podem ser analisadas?"

if "df_prepared" not in globals():
    raise ValueError("df_prepared não encontrado. Execute 02_DATA_PREPARATION.")

numeric_schema_columns = [f.name for f in df_prepared.schema.fields
                          if isinstance(f.dataType, NumericType) and not isinstance(f.dataType, BooleanType)]
configured_columns = globals().get("colunas_correlation", numeric_schema_columns)
correlation_columns = [c for c in configured_columns
                       if c in numeric_schema_columns and c not in CORRELATION_EXCLUDED_COLUMNS]
correlation_columns = list(dict.fromkeys(correlation_columns))[:CORRELATION_MAX_COLUMNS]
correlation_pairs = list(combinations(correlation_columns, 2))
print(f"Colunas numéricas selecionadas: {len(correlation_columns)}")
print(f"Pares candidatos: {len(correlation_pairs)}")
if not correlation_pairs:
    print("Sem pares numéricos suficientes: resultados vazios serão disponibilizados.")

In [0]:
# ============================================================
# 09.5 PEARSON POR PAR
# ============================================================
#
# O QUE FAZ:
# Calcula correlação linear com cobertura e variabilidade.
#
# COMO FAZ:
# Usa filtro de valores finitos e agg Spark por par.
#
# POR QUE É IMPORTANTE:
# Garante contagens e correlações sobre o mesmo subconjunto válido.
#
# PERGUNTA RESPONDIDA:
# "Qual a associação linear de cada par?"

correlation_results = []

def dq_corr_record(col_a, col_b, method, valid_n, coefficient, status):
    coefficient = dq_corr_safe_coefficient(coefficient)
    intensity, direction = dq_corr_classify(coefficient, CORRELATION_WEAK_THRESHOLD, CORRELATION_STRONG_THRESHOLD)
    return (col_a, col_b, method, int(valid_n), coefficient, status, intensity, direction)

for col_a, col_b in correlation_pairs:
    valid_pair_df = df_prepared.where(dq_corr_valid(col_a) & dq_corr_valid(col_b)).select(
        dq_corr_col(col_a).cast("double").alias("_x"),
        dq_corr_col(col_b).cast("double").alias("_y")
    )
    stats = valid_pair_df.agg(
        F.count(F.lit(1)).alias("n"),
        F.stddev_pop("_x").alias("sx"),
        F.stddev_pop("_y").alias("sy"),
        F.corr("_x", "_y").alias("r")
    ).first()  # Coleta somente uma linha de estatísticas agregadas.
    n_valid = int(stats["n"])
    applicable = (n_valid >= CORRELATION_MIN_VALID_PAIRS
                  and stats["sx"] is not None and stats["sy"] is not None
                  and stats["sx"] > 0 and stats["sy"] > 0)
    status = "APLICAVEL" if applicable and dq_corr_safe_coefficient(stats["r"]) is not None else "NAO_APLICAVEL"
    correlation_results.append(dq_corr_record(col_a, col_b, "PEARSON", n_valid,
                                               stats["r"] if status == "APLICAVEL" else None, status))

In [0]:
# ============================================================
# 09.6 SPEARMAN OPCIONAL COM TRATAMENTO DE EMPATES
# ============================================================
#
# O QUE FAZ:
# Calcula associação monotônica por postos.
#
# COMO FAZ:
# Aplica rank e contagem de empates em janelas Spark.
#
# POR QUE É IMPORTANTE:
# Complementa Pearson em relações monotônicas não lineares.
#
# PERGUNTA RESPONDIDA:
# "Os pares apresentam associação monotônica?"

if CORRELATION_ENABLE_SPEARMAN:
    print("Spearman habilitado: atenção ao custo de ordenação global por par.")
    for col_a, col_b in correlation_pairs:
        pair_df = df_prepared.where(dq_corr_valid(col_a) & dq_corr_valid(col_b)).select(
            dq_corr_col(col_a).cast("double").alias("_x"),
            dq_corr_col(col_b).cast("double").alias("_y")
        )
        # rank() devolve a primeira posição do grupo empatado.
        # posto médio = rank + (tamanho_do_empate - 1)/2.
        ranks_df = (pair_df
            .withColumn("_rx", F.rank().over(Window.orderBy("_x"))
                        + (F.count(F.lit(1)).over(Window.partitionBy("_x")) - 1) / F.lit(2.0))
            .withColumn("_ry", F.rank().over(Window.orderBy("_y"))
                        + (F.count(F.lit(1)).over(Window.partitionBy("_y")) - 1) / F.lit(2.0)))
        stats = ranks_df.agg(F.count(F.lit(1)).alias("n"),
                             F.stddev_pop("_rx").alias("sx"),
                             F.stddev_pop("_ry").alias("sy"),
                             F.corr("_rx", "_ry").alias("rho")).first()
        n_valid = int(stats["n"])
        applicable = (n_valid >= CORRELATION_MIN_VALID_PAIRS
                      and stats["sx"] is not None and stats["sy"] is not None
                      and stats["sx"] > 0 and stats["sy"] > 0
                      and dq_corr_safe_coefficient(stats["rho"]) is not None)
        correlation_results.append(dq_corr_record(
            col_a, col_b, "SPEARMAN", n_valid,
            stats["rho"] if applicable else None,
            "APLICAVEL" if applicable else "NAO_APLICAVEL"))

In [0]:
# ============================================================
# 09.7 DATAFRAME CONSOLIDADO
# ============================================================
#
# O QUE FAZ:
# Padroniza os resultados de cada par e método.
#
# COMO FAZ:
# Cria Spark DataFrame com schema explícito.
#
# POR QUE É IMPORTANTE:
# Facilita auditoria e integração com o Quality Score.
#
# PERGUNTA RESPONDIDA:
# "Quais pares e métodos foram avaliados?"

from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

profile_schema = StructType([
    StructField("coluna_a", StringType(), False),
    StructField("coluna_b", StringType(), False),
    StructField("metodo", StringType(), False),
    StructField("registros_validos", LongType(), False),
    StructField("coeficiente", DoubleType(), True),
    StructField("status", StringType(), False),
    StructField("intensidade", StringType(), False),
    StructField("direcao", StringType(), False)
])
correlation_profile_df = spark.createDataFrame(correlation_results, schema=profile_schema)
if CORRELATION_DISPLAY_RESULTS:
    display(correlation_profile_df.orderBy(F.desc(F.abs(F.col("coeficiente")))))

In [0]:
# ============================================================
# 09.8 MATRIZ LONGA DE CORRELAÇÃO
# ============================================================
#
# O QUE FAZ:
# Produz representação longa para dashboards.
#
# COMO FAZ:
# Replica os pares em ambas as direções e adiciona diagonal.
#
# POR QUE É IMPORTANTE:
# Permite construir heatmaps sem pivotar no driver.
#
# PERGUNTA RESPONDIDA:
# "Como visualizar as associações em uma matriz?"

matrix_rows = []
for a, b, method, n_valid, coef, status, intensity, direction in correlation_results:
    matrix_rows.extend([(a, b, method, coef), (b, a, method, coef)])
methods = ["PEARSON"] + (["SPEARMAN"] if CORRELATION_ENABLE_SPEARMAN else [])
for method in methods:
    for col_name in correlation_columns:
        # Diagonal meramente estrutural; não atesta variabilidade da coluna.
        matrix_rows.append((col_name, col_name, method, None))
matrix_schema = "coluna_linha STRING, coluna_coluna STRING, metodo STRING, coeficiente DOUBLE"
correlation_matrix_df = spark.createDataFrame(matrix_rows, schema=matrix_schema)
if CORRELATION_DISPLAY_RESULTS:
    display(correlation_matrix_df)

In [0]:
# ============================================================
# 09.9 ASSOCIAÇÕES FORTES / POSSÍVEL REDUNDÂNCIA
# ============================================================
#
# O QUE FAZ:
# Seleciona pares com alta correlação absoluta.
#
# COMO FAZ:
# Filtra coeficientes aplicáveis pelo limiar configurado.
#
# POR QUE É IMPORTANTE:
# Prioriza investigação sem excluir colunas automaticamente.
#
# PERGUNTA RESPONDIDA:
# "Quais pares merecem análise adicional?"

correlation_redundancy_df = correlation_profile_df.where(
    (F.col("status") == "APLICAVEL") &
    (F.abs(F.col("coeficiente")) >= F.lit(CORRELATION_STRONG_THRESHOLD))
)
if CORRELATION_DISPLAY_RESULTS:
    display(correlation_redundancy_df)

In [0]:
# ============================================================
# 09.10 RESUMO EXECUTIVO
# ============================================================
#
# O QUE FAZ:
# Consolida cobertura de execução e quantidade de sinais.
#
# COMO FAZ:
# Usa agregações Spark sobre o perfil consolidado.
#
# POR QUE É IMPORTANTE:
# Entrega visão compacta para integração futura.
#
# PERGUNTA RESPONDIDA:
# "Qual o panorama geral das correlações?"

summary_metrics = correlation_profile_df.agg(
    F.count(F.lit(1)).alias("avaliacoes"),
    F.sum(F.when(F.col("status") == "APLICAVEL", 1).otherwise(0)).alias("avaliacoes_aplicaveis"),
    F.sum(F.when(F.col("status") == "NAO_APLICAVEL", 1).otherwise(0)).alias("avaliacoes_nao_aplicaveis"),
    F.sum(F.when((F.col("status") == "APLICAVEL") &
                 (F.abs(F.col("coeficiente")) >= CORRELATION_STRONG_THRESHOLD), 1).otherwise(0)).alias("associacoes_fortes")
)
correlation_summary_df = (summary_metrics
    .withColumn("total_colunas_analisadas", F.lit(len(correlation_columns)))
    .withColumn("total_pares_candidatos", F.lit(len(correlation_pairs)))
    .withColumn("spearman_habilitado", F.lit(CORRELATION_ENABLE_SPEARMAN))
    .na.fill(0, subset=["avaliacoes_aplicaveis", "avaliacoes_nao_aplicaveis", "associacoes_fortes"]))
if CORRELATION_DISPLAY_RESULTS:
    display(correlation_summary_df)

In [0]:
# ============================================================
# 09.11 TESTES SINTÉTICOS ISOLADOS
# ============================================================
#
# O QUE FAZ:
# Verifica correlação positiva, negativa e casos constantes.
#
# COMO FAZ:
# Utiliza pequeno DataFrame Spark independente da fonte principal.
#
# POR QUE É IMPORTANTE:
# Confere pressupostos básicos sem modificar df_prepared.
#
# PERGUNTA RESPONDIDA:
# "Os cálculos se comportam conforme o esperado?"

test_df = spark.createDataFrame([(1., 2., -1., 5.), (2., 4., -2., 5.),
                                     (3., 6., -3., 5.), (4., 8., -4., 5.)],
                                    ["x", "y", "z", "constante"])
test_stats = test_df.agg(F.corr("x", "y").alias("positiva"),
                         F.corr("x", "z").alias("negativa"),
                         F.stddev_pop("constante").alias("desvio_constante")).first()
assert abs(test_stats["positiva"] - 1.0) < 1e-10
assert abs(test_stats["negativa"] + 1.0) < 1e-10
assert test_stats["desvio_constante"] == 0.0
assert correlation_profile_df.columns == [f.name for f in profile_schema.fields]
print("Testes sintéticos concluídos com sucesso.")